In [1]:
import sys
sys.path.append("../")



from corpus.jsinV3_attn_cue_multi_source import jsinV3_attn_cue_multi_source

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import src.audio_attention_transforms as aat
import src.audio_transforms as at

# these transforms take cue, foreground, background as input 
audio_transforms = aat.AudioCompose([
    aat.AudioToTensor(),
    aat.RMSNormalizeForegroundAndBackground(rms_level=0.1), # normalize so all signals at same level pre-mix
    aat.CombineWithRandomDBSNR(low_snr=-20, high_snr=20),
    aat.RMSNormalizeMixtureAndMatchCueLevel(rms_level=0.1), # set cue to same level as target 
#     aat.UnsqueezeAudio(dim=0),
])
# these transforms take foreground, background as input 
bg_combine_transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), # set distractors to same level for matched cue level training  
                # at.CombineWithRandomDBSNR(low_snr=config['noise_kwargs']['low_snr'], high_snr=config['noise_kwargs']['high_snr']),
                at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
                # at.UnsqueezeAudio(dim=0),
            ])

In [3]:
root = '/om2/user/msaddler/projects/ibmHearingAid/assets/data/datasets/JSIN_v3.00/nStim_20000/2000ms/rms_0.1/noiseSNR_-10_10/stimSR_20000/reverb_none/noise_all/JSIN_all_v3/subsets/'


dataset = jsinV3_attn_cue_multi_source(root=root, 
                                       mode='train',
                                       transform=[audio_transforms,bg_combine_transforms], 
                                       with_audioset=True,
                                       demo=True,
                                       n_talkers=1)

In [4]:


foreground, background, noise, signal, fg_cue, fg_target = dataset[0]


In [5]:
noise

array([-0.86587197, -0.8041066 , -0.7731805 , ...,  0.37367198,
        1.0712292 ,  1.0763246 ], dtype=float32)

In [6]:
foreground.shape

(40000,)

In [7]:
from IPython.display import Audio


In [8]:
sr = 20_000

In [9]:
Audio(noise, rate=sr)

In [10]:
Audio(background, rate=sr)